# 01 Egyptian Food Cleaning

Clean the Egyptian Food dataset for the MVP Food Knowledge Base.

Output: `data/processed/egyptian_food_clean.parquet`

In [ ]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW_PATH = ROOT / 'Egyptian Food.csv'
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'egyptian_food_clean.parquet'

FINAL_COLUMNS = [
    'food_id', 'food_name_en', 'food_name_ar', 'food_group', 'meal_types',
    'serving_name', 'serving_weight_g', 'nutrition_basis', 'calories',
    'protein', 'carbs', 'fat', 'fiber', 'diet_tags', 'allergens',
    'is_composite_dish', 'source'
]
TARGET_GROUPS = {
    'Grains', 'Vegetables', 'Fruits', 'Protein', 'Dairy', 'Legumes',
    'Healthy Fats', 'Composite Dish', 'Beverages', 'Snacks'
}

def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    for encoding in ('utf-8-sig', 'utf-8', 'latin1'):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding='latin1', low_memory=False)

def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    text = text.replace('/', ' with ')
    text = re.sub(r'\s*,\s*', ', ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('barlcy', 'barley')
    text = text.replace('kernels  on', 'kernels on')
    return text.strip(' ,')

def stable_id(prefix: str, name: str) -> str:
    digest = hashlib.sha1(name.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{digest}'

def to_number(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype('string')
        .str.strip()
        .str.replace(',', '', regex=False)
        .str.replace(r'^(t|trace)$', '0', case=False, regex=True)
        .str.replace(r'[^0-9.+\-eE]', '', regex=True)
    )
    return pd.to_numeric(cleaned, errors='coerce')

def is_composite(name: str) -> bool:
    markers = [' with ', 'cooked', 'fried', 'grilled', 'stuffed', 'soup', 'stew', 'sandwich', 'pizza', 'pie', 'omelet']
    return any(marker in name for marker in markers)

def contains_word(name: str, terms: list[str]) -> bool:
    tokens = set(re.findall(r'[a-z]+', name.lower()))
    return any(term in tokens for term in terms)

def infer_food_group(name: str) -> str:
    if is_composite(name):
        return 'Composite Dish'
    if contains_word(name, ['rice', 'corn', 'barley', 'wheat', 'macaroni', 'noodle', 'noodles', 'bread', 'flour', 'grain', 'grains', 'pasta', 'semolina']):
        return 'Grains'
    if contains_word(name, ['bean', 'beans', 'lentil', 'lentils', 'chickpea', 'chickpeas', 'pea', 'peas', 'fava', 'ful', 'cowpeas', 'lupines', 'soybeans']):
        return 'Legumes'
    if contains_word(name, ['milk', 'cheese', 'yogurt', 'yoghurt', 'cream', 'butter']):
        return 'Dairy'
    if contains_word(name, ['meat', 'beef', 'veal', 'lamb', 'chicken', 'duck', 'turkey', 'fish', 'egg', 'eggs', 'liver', 'kidney', 'crab', 'bream', 'mullet', 'herring', 'catfish', 'mackerel', 'sardine', 'sardines', 'shrimp', 'sole', 'tilapia', 'tuna', 'gelatin']):
        return 'Protein'
    if contains_word(name, ['oil', 'ghee', 'fat', 'margarine', 'almonds', 'hazelnuts', 'nuts', 'pinenuts', 'peanuts', 'nutmeg', 'sesame', 'walnuts', 'mayonnaise', 'avocado', 'coconuls']):
        return 'Healthy Fats'
    if contains_word(name, ['juice', 'tea', 'coffee', 'drink', 'drinks', 'beverage', 'beer', 'water', 'nectar', 'crabonated', 'carbonated']):
        return 'Beverages'
    if contains_word(name, ['apple', 'apples', 'banana', 'bananas', 'orange', 'oranges', 'date', 'dates', 'grape', 'grapes', 'grapea', 'grapefruis', 'fig', 'figs', 'melon', 'melons', 'watermelons', 'fruit', 'apricots', 'cantaloupe', 'guavas', 'lemons', 'lime', 'mangos', 'mulberry', 'jujuba', 'kiwifruit', 'loquate', 'mandarin', 'papaya', 'peaches', 'pears', 'persimmon', 'pineapple', 'plums', 'pomegranate', 'prunes', 'raisins', 'raspberries', 'strawberries', 'tamarind', 'sycamore']):
        return 'Fruits'
    if contains_word(name, ['tomato', 'tomatoes', 'onion', 'onions', 'potato', 'potatoes', 'sweetpotatos', 'carrot', 'carrots', 'cucumber', 'vegetable', 'vegetables', 'spinach', 'molokhia', 'artichokes', 'beetroots', 'broccoli', 'brussels', 'cabbage', 'cauliflower', 'celery', 'chard', 'chicory', 'colocasia', 'coriander', 'dill', 'eggplant', 'fennel', 'fenugreek', 'garlic', 'ginger', 'leeks', 'lettuce', 'mallow', 'mashrooms', 'mushrooms', 'mint', 'okra', 'olives', 'parsley', 'pepper', 'peppers', 'pumpkin', 'purslane', 'radishes', 'rocket', 'squash', 'thyme', 'turnips']):
        return 'Vegetables'
    if contains_word(name, ['cake', 'cakes', 'sweet', 'sweets', 'sweeteners', 'sugar', 'biscuit', 'chocolate', 'halawa', 'jam', 'jams', 'candy', 'gum', 'honey', 'molasses', 'pastry', 'jelly', 'baklava', 'basbusa', 'oriental']):
        return 'Snacks'
    return 'Composite Dish'

ARABIC_TERMS = {
    'rice': 'أرز', 'bread': 'عيش', 'barley': 'شعير', 'corn': 'ذرة', 'macaroni': 'مكرونة',
    'milk': 'لبن', 'cheese': 'جبنة', 'yogurt': 'زبادي', 'meat': 'لحمة', 'beef': 'لحم بقري',
    'chicken': 'فراخ', 'fish': 'سمك', 'egg': 'بيض', 'bean': 'فول', 'lentil': 'عدس',
    'oil': 'زيت', 'tomato': 'طماطم', 'potato': 'بطاطس', 'onion': 'بصل', 'apple': 'تفاح',
    'banana': 'موز', 'orange': 'برتقال', 'juice': 'عصير', 'fried': 'مقلي', 'grilled': 'مشوي',
    'cooked': 'مطبوخ', 'raw': 'نيء', 'with': 'مع'
}

def generate_arabic_name(name: str) -> str:
    translated = []
    for token in re.findall(r'[a-z]+', name.lower()):
        if token in ARABIC_TERMS:
            translated.append(ARABIC_TERMS[token])
    if translated:
        return ' '.join(dict.fromkeys(translated))
    return ''

def infer_meal_types(group: str, name: str) -> str:
    if group in {'Grains', 'Dairy', 'Legumes'}:
        return 'Breakfast|Lunch|Dinner'
    if group in {'Protein', 'Composite Dish'}:
        return 'Lunch|Dinner'
    if group in {'Fruits', 'Snacks', 'Beverages'}:
        return 'Snack'
    return 'Lunch|Dinner'

def infer_allergens(name: str) -> str:
    allergens = []
    if any(k in name for k in ['milk', 'cheese', 'yogurt', 'cream', 'butter']):
        allergens.append('milk')
    if 'egg' in name:
        allergens.append('egg')
    if any(k in name for k in ['fish', 'shrimp']):
        allergens.append('fish')
    if any(k in name for k in ['wheat', 'bread', 'macaroni', 'noodle', 'flour']):
        allergens.append('wheat')
    return '|'.join(allergens)

def infer_diet_tags(group: str, name: str) -> str:
    tags = []
    if group in {'Grains', 'Vegetables', 'Fruits', 'Legumes', 'Healthy Fats', 'Beverages'} and not any(k in name for k in ['meat', 'chicken', 'fish', 'egg', 'milk', 'cheese', 'yogurt']):
        tags.append('plant_based')
    if group in {'Protein', 'Dairy'}:
        tags.append('animal_based')
    if group == 'Composite Dish':
        tags.append('composite')
    return '|'.join(tags)

raw = read_csv_with_fallback(RAW_PATH)
raw = raw.dropna(axis=1, how='all')
raw.columns = [re.sub(r'[^a-z0-9]+', '_', c.strip().lower()).strip('_') for c in raw.columns]

column_map = {
    'food': 'food_name_en',
    'energy_kcal': 'calories',
    'protein_g': 'protein',
    'fat_g': 'fat',
    'fiber_g': 'fiber',
    'carbohydrate_g': 'carbs',
}
raw = raw.rename(columns={k: v for k, v in column_map.items() if k in raw.columns})

required_nutrients = ['calories', 'protein', 'carbs', 'fat', 'fiber']
for col in required_nutrients:
    if col in raw.columns:
        raw[col] = to_number(raw[col])
    else:
        raw[col] = np.nan

clean = raw[['food_name_en'] + required_nutrients].copy()
clean['food_name_en'] = clean['food_name_en'].map(normalize_text)
clean = clean[clean['food_name_en'].ne('')].copy()
clean['food_group'] = clean['food_name_en'].map(infer_food_group)

for col in required_nutrients:
    group_median = clean.groupby('food_group')[col].transform('median')
    clean[col] = clean[col].fillna(group_median)
    fallback = 0.0 if col == 'fiber' else clean[col].median()
    clean[col] = clean[col].fillna(fallback).clip(lower=0)

plausible_limits = {'calories': 950, 'protein': 100, 'carbs': 100, 'fat': 100, 'fiber': 80}
for col, max_value in plausible_limits.items():
    invalid = clean[col].gt(max_value)
    clean.loc[invalid, col] = np.nan
    group_median = clean.groupby('food_group')[col].transform('median')
    clean[col] = clean[col].fillna(group_median)
    fallback = 0.0 if col == 'fiber' else clean[col].median()
    clean[col] = clean[col].fillna(fallback).clip(lower=0, upper=max_value)

macro_cols = ['protein', 'carbs', 'fat', 'fiber']
invalid_macro_sum = clean[macro_cols].sum(axis=1).gt(120)
clean.loc[invalid_macro_sum, macro_cols] = np.nan
for col in macro_cols:
    group_median = clean.groupby('food_group')[col].transform('median')
    clean[col] = clean[col].fillna(group_median)
    fallback = 0.0 if col == 'fiber' else clean[col].median()
    clean[col] = clean[col].fillna(fallback).clip(lower=0, upper=plausible_limits[col])

clean['name_key'] = clean['food_name_en'].str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
clean['completeness'] = clean[required_nutrients].notna().sum(axis=1)
duplicate_count = int(clean.duplicated('name_key').sum())
clean = clean.sort_values(['name_key', 'completeness'], ascending=[True, False]).drop_duplicates('name_key', keep='first')

clean['food_id'] = clean['name_key'].map(lambda value: stable_id('egy', value))
clean['food_name_ar'] = clean['food_name_en'].map(generate_arabic_name)
clean['meal_types'] = [infer_meal_types(group, name) for group, name in zip(clean['food_group'], clean['food_name_en'])]
clean['serving_name'] = '100 g'
clean['serving_weight_g'] = 100.0
clean['nutrition_basis'] = 'per_100g'
clean['diet_tags'] = [infer_diet_tags(group, name) for group, name in zip(clean['food_group'], clean['food_name_en'])]
clean['allergens'] = clean['food_name_en'].map(infer_allergens)
clean['is_composite_dish'] = clean['food_name_en'].map(is_composite)
clean['source'] = 'egyptian_food'

clean = clean[FINAL_COLUMNS].copy()
clean.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')

print(f'Raw rows: {len(raw)}')
print(f'Duplicate Egyptian food names detected: {duplicate_count}')
print(f'Clean rows exported: {len(clean)}')
print(f'Output: {OUTPUT_PATH}')
print(clean.head(10))
